# 02 学習（Phase 3：CMA-ES）
長時間の学習は **ターミナルで CLI を回す**（ノートブックはカーネルが落ちると止まるため）。
スリープ防止付きで実行する例（`runs/build-001` は任意の名前）:

```bash
cd training/puyo-lv5
caffeinate -i python3 py/train.py --run runs/build-001 --gens 100
```

中断しても `--resume` で続きから再開できる（世代ごとに `state.json` を保存）。
このノートブックは進捗の可視化用。

In [ ]:
import sys, subprocess, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "py"))
import numpy as np
import puyosim as ps

import csv
import matplotlib.pyplot as plt
RUN = ROOT / "runs" / "build-001"   # ← 見たい run に変える

In [ ]:
rows = list(csv.DictReader(open(RUN / "log.csv")))
gen = [int(r["gen"]) for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(gen, [float(r["wr_mean"]) for r in rows], label="候補の平均勝率")
ax[0].plot(gen, [float(r["wr_best"]) for r in rows], label="候補の最高勝率", alpha=0.5)
ev = [(int(r["gen"]), float(r["eval_wr"])) for r in rows if r["eval_wr"]]
if ev:
    ax[0].plot(*zip(*ev), "o-", label="平均ベクトル vs 現行（定期評価）")
ax[0].axhline(0.5, color="gray", lw=0.8)
ax[0].set_xlabel("世代"); ax[0].set_ylabel("勝率"); ax[0].legend()
ax[1].plot(gen, [float(r["sigma"]) for r in rows]); ax[1].set_xlabel("世代"); ax[1].set_ylabel("sigma")
plt.show()

In [ ]:
# パラメータ（正規化座標 0..1）の推移
import params as P
X = np.array([json.loads(r["mean_x"]) for r in rows])
fig, ax = plt.subplots(figsize=(12, 6))
for i, p in enumerate(P.SPACE):
    ax.plot(gen, X[:, i], label=p.name)
ax.legend(ncol=3, fontsize=7); ax.set_xlabel("世代"); ax.set_ylabel("正規化値")
plt.show()

In [ ]:
best = json.loads((RUN / "best.json").read_text())
print(f"best: gen={best['gen']} 定期評価勝率={best['winRate']:.3f}（選抜に使った評価なので過大。最終判断は 03 の検証で）")
for k, v in best["changed"].items():
    print(f"  {k:28s} {v['from']} → {v['to']}")